# MemSysExplorer Profiler Demo

This notebook demonstrates how to use the MemSysExplorer profiling system with different profilers.

**Profilers covered:**
1. DynamoRIO - Dynamic Binary Instrumentation
2. Sniper - Architectural Simulator
3. Perf - Hardware Performance Counters (CPU)

---
## 1. Main Interface Help

The `main.py` script is the entry point for all profilers.

In [ ]:
!python3 main.py --help

---
## 2. Sample Test Program

We use a simple test program that performs a loop with sqrt calculations.

In [21]:
!cat test/sample.c

#include <stdio.h>
#include <stdlib.h>
#include <math.h>

int main() {
    volatile double x = 0.0;

    for (long i = 0; i < 1000000; i++) {
        x += sqrt(i);
    }

    printf("Done: %f\n", x);
    return 0;
}

In [ ]:
# Build the sample program (if not already built)
!cd test && make

---
---
# DynamoRIO Profiler

DynamoRIO is a Dynamic Binary Instrumentation (DBI) tool that tracks memory access patterns without modifying source code.

## 3.1 DynamoRIO Configuration

The memcount client uses a text-based configuration file.

In [22]:
!cat config/memcount_config.txt

# MemCount Configuration File
# This file controls the behavior of the memcount DynamoRIO client
# Format: key=value (one per line)

# Cache and Memory Parameters
cache_line_size=64
hll_bits=8
sample_hll_bits=8
sample_window_refs=2000
max_mem_refs=8192

# Output Control
enable_trace=false
wss_stat_tracking=false

# WSS Tracking Methods
wss_exact_tracking=true
wss_hll_tracking=false

# Instruction Threshold Control
# Set enable_instruction_threshold=true to terminate after a specific number of instructions
# Useful for testing or limiting profiling to a specific instruction count
enable_instruction_threshold=false
instruction_threshold=100000000

# Protobuf Output Files (base names, process ID will be appended)
pb_trace_output=memtrace
pb_timeseries_output=timeseries


## 3.2 Run DynamoRIO Profiler

**Common Command:**
```bash
python3 main.py --profiler dynamorio --action both --executable <your_executable>
```

In [ ]:
# Clean up previous outputs before DynamoRIO run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_dynamorio*.json

In [23]:
# Run DynamoRIO profiler with the sample program
!python3 main.py --profiler dynamorio --action both --config config/memcount_config.txt --executable test/sample

Registered Profilers: ['dynamorio', 'sniper']
Registered PatternConfig Classes: ['dynamorio', 'sniper']
Registered Metadata for: dynamorio
Registered Metadata for: sniper
Registered Metadata Classes: ['dynamorio', 'sniper']
Executing: /home/dnguye/projects/MemSysExplorer/apps/profilers/dynamorio/dynamorio_install/DynamoRIO-11.3.0-1/build/bin64/drrun -c /home/dnguye/projects/MemSysExplorer/apps/profilers/dynamorio/build/libmemcount.so -config config/memcount_config.txt -- test/sample
Profiling completed successfully.
Done: 666666166.458842
Instrumentation results:
  saw 2058544 memory references
  number of reads: 1042026
  number of writes: 1016518
  working set size: 2779
  execution time (us): 201739
  execution time (ms): 201.739
  execution time (s): 0.201739

Read size breakdown:
  1-byte reads: 12504
  2-byte reads: 1103
  4-byte reads: 6279
  8-byte reads: 1021661
  16-byte reads: 475
  32-byte reads: 4
  64-byte reads: 0
  other-size reads: 0

Write size breakdown:
  1-byte wri

## 3.3 DynamoRIO Output

View the generated PatternConfig output.

In [24]:
import json
import glob
import os

# Find DynamoRIO pattern config (named after executable: sample)
drio_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if drio_files:
    with open(drio_files[0]) as f:
        data = json.load(f)
    print(f"File: {drio_files[0]}")
    print(json.dumps(data, indent=2))
else:
    print("No pattern config found. Run the profiler first.")

File: memsyspatternconfig_sample.json
{
  "read_size_histogram": {
    "1": 12504,
    "2": 1103,
    "4": 6279,
    "8": 1021661,
    "16": 475,
    "32": 4,
    "64": 0,
    "other": 0
  },
  "write_size_histogram": {
    "1": 225,
    "2": 26,
    "4": 1763,
    "8": 1013825,
    "16": 679,
    "32": 0,
    "64": 0,
    "other": 0
  },
  "exp_name": "DynamoRIOProfilers",
  "benchmark_name": " ",
  "read_freq": 5.165218425787775,
  "total_reads": 1042026,
  "write_freq": 5.038777826796009,
  "total_writes": 1016518,
  "read_size": 8,
  "write_size": 8,
  "total_writes_i": 1,
  "total_writes_d": 1,
  "total_reads_d": 1,
  "total_reads_i": 1,
  "workingset_size": 2779,
  "execution_time": 201739,
  "peak_memory_kb": null,
  "metadata": null,
  "unit": {
    "read_freq": "accesses/us",
    "write_freq": "accesses/us",
    "total_reads": "count",
    "total_writes": "count",
    "workingset_size": "count",
    "execution_time": "microseconds",
    "peak_memory_kb": "KB"
  }
}


## 3.4 DynamoRIO Metadata

View the system metadata captured during profiling.

In [ ]:
import json
import glob
import os

# Find DynamoRIO metadata
meta_files = sorted(glob.glob("memsysmetadata_dynamorio*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))

File: memsysmetadata_dynamorio.json
{
  "gpu_name": "NVIDIA GeForce RTX 4080 SUPER",
  "driver_version": "560.94",
  "gpu_memory_MB": 16376,
  "cpu_info": {
    "Architecture": "x86_64",
    "CPU op-mode(s)": "32-bit, 64-bit",
    "Address sizes": "39 bits physical, 48 bits virtual",
    "Byte Order": "Little Endian",
    "CPU(s)": "8",
    "On-line CPU(s) list": "0-7",
    "Vendor ID": "GenuineIntel",
    "Model name": "Intel(R) Core(TM) i7-14700KF",
    "CPU family": "6",
    "Model": "183",
    "Thread(s) per core": "2",
    "Core(s) per socket": "4",
    "Socket(s)": "1",
    "Stepping": "1",
    "BogoMIPS": "6835.19",
    "Flags": "fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology tsc_reliable nonstop_tsc cpuid pni pclmulqdq vmx ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt tsc_deadline_timer aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch 

---
---
# Sniper Profiler

Sniper is a cycle-accurate architectural simulator that models multi-core systems with detailed memory hierarchy.

## 4.1 Sniper Configuration

Sniper uses `.cfg` files to define CPU and memory hierarchy parameters. The `gainestown` config models an Intel Xeon X5550.

In [26]:
!cat config/gainestown.cfg

# Configuration file for Xeon X5550 Gainestown
# See http://en.wikipedia.org/wiki/Gainestown_(microprocessor)#Gainestown
# and http://ark.intel.com/products/37106

#include nehalem

[perf_model/core]
frequency = 2.66

[perf_model/l2_cache]
# Enable L2 cache tracing
trace_enabled = true
trace_file = cache_trace.out
trace_buffer_size = 1024

[perf_model/l3_cache]
perfect = false
cache_block_size = 64
cache_size = 8192
associativity = 16
address_hash = mask
replacement_policy = lru
data_access_time = 30 # 35 cycles total according to membench, +L1+L2 tag times
tags_access_time = 10
perf_model_type = parallel
writethrough = 0
shared_cores = 4
# Enable L3 cache tracing  
trace_enabled = true
trace_file = cache_trace.out
trace_buffer_size = 1024

[perf_model/dram_directory]
# total_entries = number of entries per directory controller.
total_entries = 1048576
associativity = 16
directory_type = full_map

[perf_model/dram]
# -1 means that we have a number of distributed DRAM controllers (4 in 

## 4.2 Run Sniper Profiler

**Common Command:**
```bash
python3 main.py -p sniper -a both --config gainestown --level l3 --results_dir . --executable <your_executable>
```

In [27]:
# Clean up previous outputs before Sniper run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_sniper*.json

In [28]:
# Run Sniper profiler with the sample program
!python3 main.py -p sniper -a both --config gainestown --level l3 --results_dir . --executable test/sample

Registered Profilers: ['dynamorio', 'sniper']
Registered PatternConfig Classes: ['dynamorio', 'sniper']
Registered Metadata for: dynamorio
Registered Metadata for: sniper
Registered Metadata Classes: ['dynamorio', 'sniper']
Executing: /home/dnguye/projects/MemSysExplorer/apps/profilers/sniper/snipersim/run-sniper -c gainestown -- test/sample
[SNIPER] Start
[SNIPER] --------------------------------------------------------------------------------
[SNIPER] Sniper using SIFT/trace-driven frontend
[SNIPER] Running full application in DETAILED mode
[SNIPER] --------------------------------------------------------------------------------
[SNIPER] Enabling performance models
[SNIPER] Setting instrumentation mode to DETAILED
[RECORD-TRACE] Using the Pin frontend (sift/recorder)
Done: 666666166.458842
[TRACE:0] -- DONE --
[SNIPER] Disabling performance models
[SNIPER] Leaving ROI after 8.80 seconds
[SNIPER] Simulated 11.2M instructions, 39.5M cycles, 0.28 IPC
[SNIPER] Simulation speed 1270.7 KIP

## 4.3 Sniper Output

View the generated PatternConfig output. Sniper generates per-core statistics.

In [ ]:
import json
import glob
import os

# Find Sniper pattern config (named after executable: sample)
# Sniper runs after DynamoRIO, so look for most recent sample*.json
sniper_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if sniper_files:
    with open(sniper_files[0]) as f:
        data = json.load(f)
    print(f"File: {sniper_files[0]}")
    print(json.dumps(data, indent=2))

File: memsyspatternconfig_sample.json
[
  {
    "read_size_histogram": {
      "1": 0,
      "2": 0,
      "4": 0,
      "8": 0,
      "16": 0,
      "32": 0,
      "64": 2558,
      "other": 0
    },
    "write_size_histogram": {
      "1": 0,
      "2": 0,
      "4": 0,
      "8": 0,
      "16": 0,
      "32": 0,
      "64": 648,
      "other": 0
    },
    "load_hits": 2558,
    "load_misses": 0,
    "store_hits": 648,
    "store_misses": 0,
    "exp_name": "SniperProfilers",
    "benchmark_name": "core_0",
    "read_freq": 31580246913.58025,
    "total_reads": 2558,
    "write_freq": 8000000000.0,
    "total_writes": 648,
    "read_size": 64,
    "write_size": 64,
    "total_writes_i": 1,
    "total_writes_d": 1,
    "total_reads_d": 1,
    "total_reads_i": 1,
    "workingset_size": 0,
    "execution_time": 8.1e-08,
    "peak_memory_kb": null,
    "metadata": null,
    "unit": {
      "read_freq": "count/s",
      "write_freq": "count/s",
      "total_reads": "count",
      "total_

## 4.4 Sniper Metadata

View the system metadata captured during simulation.

In [ ]:
import json
import glob
import os

# Find Sniper metadata
meta_files = sorted(glob.glob("memsysmetadata_sniper*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))

File: memsysmetadata_sniper.json
{
  "gpu_name": "NVIDIA GeForce RTX 4080 SUPER",
  "driver_version": "560.94",
  "gpu_memory_MB": 16376,
  "cpu_info": {
    "Architecture": "x86_64",
    "CPU op-mode(s)": "32-bit, 64-bit",
    "Address sizes": "39 bits physical, 48 bits virtual",
    "Byte Order": "Little Endian",
    "CPU(s)": "8",
    "On-line CPU(s) list": "0-7",
    "Vendor ID": "GenuineIntel",
    "Model name": "Intel(R) Core(TM) i7-14700KF",
    "CPU family": "6",
    "Model": "183",
    "Thread(s) per core": "2",
    "Core(s) per socket": "4",
    "Socket(s)": "1",
    "Stepping": "1",
    "BogoMIPS": "6835.19",
    "Flags": "fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology tsc_reliable nonstop_tsc cpuid pni pclmulqdq vmx ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt tsc_deadline_timer aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch inv

---
---
# Perf Profiler

Perf leverages **Hardware Performance Counters (HPC)** to collect low-level hardware metrics from CPUs. It provides insights into cache behavior at different memory hierarchy levels.

**Note:** Perf requires Linux with access to hardware performance counters. It may not work in WSL or virtualized environments.

## 5.1 Perf Profiler Help

View the available options for the Perf profiler.

In [31]:
!python3 main.py --profiler perf --action both --help

Registered Profilers: ['dynamorio', 'sniper']
Registered PatternConfig Classes: ['dynamorio', 'sniper']
Registered Metadata for: dynamorio
Registered Metadata for: sniper
Registered Metadata Classes: ['dynamorio', 'sniper']
usage: main.py [-h] -p PROFILER -a {profiling,extract_metrics,both}

Profiling Tool Driver

options:
  -h, --help            show this help message and exit
  -p PROFILER, --profiler PROFILER
                        Select the profiler to use.
  -a {profiling,extract_metrics,both}, --action {profiling,extract_metrics,both}


## 5.2 Run Perf Profiler

**Common Command:**
```bash
python3 main.py -p perf -a both --level l2 --executable <your_executable>
```

**Note:** May require `sudo sh -c 'echo -1 > /proc/sys/kernel/perf_event_paranoid'` for HPC access.

In [ ]:
# Clean up previous outputs before Perf run
!rm -f memsyspatternconfig_sample*.json memsysmetadata_perf*.json

In [ ]:
# Run Perf profiler with the sample program (L2 cache level)
!python3 main.py -p perf -a both --level l2 --executable test/sample

## 5.3 Perf Output

View the generated PatternConfig output.

In [ ]:
import json
import glob
import os

# Find Perf pattern config (named after executable: sample)
perf_files = sorted(glob.glob("memsyspatternconfig_sample*.json"), key=os.path.getmtime, reverse=True)
if perf_files:
    with open(perf_files[0]) as f:
        data = json.load(f)
    print(f"File: {perf_files[0]}")
    print(json.dumps(data, indent=2))

## 5.4 Perf Metadata

View the system metadata captured during profiling.

In [ ]:
import json
import glob
import os

# Find Perf metadata
meta_files = sorted(glob.glob("memsysmetadata_perf*.json"), key=os.path.getmtime, reverse=True)
if meta_files:
    with open(meta_files[0]) as f:
        data = json.load(f)
    print(f"File: {meta_files[0]}")
    print(json.dumps(data, indent=2))